# Web Scraping News Articles

#### This notebook scrapes news articles related to the 2024 Bangladesh election from multiple news outlets and stores the raw article text in a CSV file.
#### The scraped data will later be translated and analyzed using AWS services.

In [1]:
# Import required libraries for web scraping
print("📚 Setting up the environment...")


import pandas as pd
import requests
from bs4 import BeautifulSoup
import time

print("✅ Libraries imported successfully")

📚 Setting up the environment...
✅ Libraries imported successfully


In [2]:
df = pd.read_csv("../data/articles.csv", encoding='utf-8')
df.head()

,Source,Title,URL,Language,Unnamed: 4
0,The Daily Star,New Polls Timing BNP upbeat process irks Jamaa...,https://www.thedailystar.net/news/bangladesh/p...,en,NaN
1,The Daily Star,BNP announces candidates for 36 more constitue...,https://www.thedailystar.net/news/bangladesh/p...,en,NaN
2,The Daily Star,Seat-sharing snub riles BNP allies,https://www.thedailystar.net/news/bangladesh/p...,en,NaN
3,The Daily Star,Seat sharing BNP still keeps allies hanging,https://www.thedailystar.net/news/bangladesh/p...,en,NaN
4,The Daily Star,"Election to be tough for BNP, bigger test if v...",https://www.thedailystar.net/news/bangladesh/p...,en,NaN


In [3]:
# Clean and standardize column names
df = df.rename(columns={
    "Source": "source",
    "URL": "url",
    "Language": "language",
    "Title": "title"
})

# Drop unnecessary column created by Excel
df = df.drop(columns=["Unnamed: 4"])

df.head()


,source,title,url,language
0,The Daily Star,New Polls Timing BNP upbeat process irks Jamaa...,https://www.thedailystar.net/news/bangladesh/p...,en
1,The Daily Star,BNP announces candidates for 36 more constitue...,https://www.thedailystar.net/news/bangladesh/p...,en
2,The Daily Star,Seat-sharing snub riles BNP allies,https://www.thedailystar.net/news/bangladesh/p...,en
3,The Daily Star,Seat sharing BNP still keeps allies hanging,https://www.thedailystar.net/news/bangladesh/p...,en
4,The Daily Star,"Election to be tough for BNP, bigger test if v...",https://www.thedailystar.net/news/bangladesh/p...,en


In [4]:
# Scrape Daily Star articles

daily_star_texts = []
daily_star_urls = []

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_10_1) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/39.0.2171.95 Safari/537.36"
    )
}

# Loop over Daily Star URLs
for url in df["url"][:10]:
    try:
        response = requests.get(url, headers=headers, timeout=10)

        if response.status_code != 200:
            print(f"❌ Failed to fetch {url} (status {response.status_code})")
            continue

        soup = BeautifulSoup(response.content, "html.parser")

        article = soup.find("article")
        if not article:
            print(f"❌ No article body found for {url}")
            continue

        paragraphs = article.find_all("p")
        text = "\n".join(p.get_text(strip=True) for p in paragraphs)

        daily_star_texts.append(text)
        daily_star_urls.append(url)

        print(f"✅ Scraped Daily Star article: {url}")

    except Exception as e:
        print(f"❌ Error scraping {url}: {str(e)}")


✅ Scraped Daily Star article: https://www.thedailystar.net/news/bangladesh/politics/news/new-polls-timing-bnp-upbeat-process-irks-jamaat-ncp-3916976
✅ Scraped Daily Star article: https://www.thedailystar.net/news/bangladesh/politics/news/bnp-announces-candidates-36-more-constituencies-4050536
✅ Scraped Daily Star article: https://www.thedailystar.net/news/bangladesh/politics/news/seat-sharing-snub-riles-bnp-allies-4052206
✅ Scraped Daily Star article: https://www.thedailystar.net/news/bangladesh/politics/news/seat-sharing-bnp-still-keeps-allies-hanging-4062181
✅ Scraped Daily Star article: https://www.thedailystar.net/news/bangladesh/politics/news/election-be-tough-bnp-bigger-test-if-voted-power-4061006
✅ Scraped Daily Star article: https://www.thedailystar.net/opinion/views/news/old-habits-die-hard-bnps-response-4033496
✅ Scraped Daily Star article: https://www.thedailystar.net/news/bangladesh/politics/news/bnp-47-caught-between-prospects-and-perils-3975581
✅ Scraped Daily Star articl

In [5]:
# check if it worked
print(daily_star_texts[0][:500]) # 0 for the first article
print(daily_star_texts[3][:500]) # 3 for the third article

The interim government's revised election timeline with certain conditions has stirred cautious optimism as well as raised questions among  political parties.
While many parties including the BNP have welcomed the proposed  timeline -- mid-February next year --  concerns have emerged regarding how the decision was reached through discussions with only one party.
The BNP had long been demanding that the election be held by the year-end. It had been criticising the interim government for not unvei
Even though a week has passed since the announcement of the election schedule, BNP allies still do not know how many seats they might be allotted.
They said frustration is deepening within their ranks as the BNP has kept the seat-sharing issue hanging for so long.
The parties that were part of the simultaneous movement with the BNP also claimed the BNP did not hold any discussions with them before unveiling its nominees in most constituencies.
In efforts to assuage the allies, the BNP held one-

In [7]:
# Scrape Prothom Alo articles

prothom_alo_texts = []
prothom_alo_urls = []
prothom_alo_languages = []

# Loop over Prothom Alo URLs
for url in df["url"][10:]:
    try:
        response = requests.get(url, headers=headers, timeout=10)

        if response.status_code != 200:
            print(f"❌ Failed to fetch {url} (status {response.status_code})")
            continue

        soup = BeautifulSoup(response.content, "html.parser")

        body = soup.find_all("div", class_="story-element-text")
        if not body:
            print(f"❌ No article body found for {url}")
            continue

        text = "\n".join(p.get_text(strip=True) for p in body)

        prothom_alo_texts.append(text)
        prothom_alo_urls.append(url)

        # Detect language from URL (Bangla vs English)
        if "bangla" in url or "/bn/" in url:
            prothom_alo_languages.append("bn")
        else:
            prothom_alo_languages.append("en")

        print(f"✅ Scraped Prothom Alo article: {url}")

    except Exception as e:
        print(f"❌ Error scraping {url}: {str(e)}")


✅ Scraped Prothom Alo article: https://en.prothomalo.com/bangladesh/politics/n8wdif39or
✅ Scraped Prothom Alo article: https://en.prothomalo.com/bangladesh/9pkzu86yuv
✅ Scraped Prothom Alo article: https://en.prothomalo.com/bangladesh/politics/pbdxuupv0a
✅ Scraped Prothom Alo article: https://en.prothomalo.com/bangladesh/jsto3ql5zk
✅ Scraped Prothom Alo article: https://en.prothomalo.com/bangladesh/politics/kxnoeb9umk
✅ Scraped Prothom Alo article: https://en.prothomalo.com/bangladesh/politics/dunevrwld4
✅ Scraped Prothom Alo article: https://en.prothomalo.com/bangladesh/politics/k4see0a2ji
✅ Scraped Prothom Alo article: https://en.prothomalo.com/bangladesh/politics/z9eceb7lr1
✅ Scraped Prothom Alo article: https://en.prothomalo.com/bangladesh/politics/fx9ahjguud
✅ Scraped Prothom Alo article: https://en.prothomalo.com/opinion/op-ed/pmyhyeikw9
✅ Scraped Prothom Alo article: https://www.prothomalo.com/politics/juller8ad4
✅ Scraped Prothom Alo article: https://www.prothomalo.com/banglade

In [8]:
# check if it worked
print(prothom_alo_texts[0][:500])
print(prothom_alo_texts[5][:500])
print(prothom_alo_texts[15][:500])

The BNP’s earlier thinking on forming an alliance or contesting elections jointly with its partners in the simultaneous movement is undergoing some change.Relevant responsible sources say the BNP will not form an electoral alliance with its partner parties in the election. There will, however, be seat-sharing arrangements in some constituencies. As part of this, the BNP will not nominate candidates in the constituencies of several top leaders of allied parties. In addition, if the BNP has alread
In the chilly winter weather, electioneering has gained momentum in Habiganj’s four constituencies ahead of the 13th parliamentary election. Nominated candidates from BNP, Jamaat-e-Islami, and other parties are taking part daily in rallies, processions, courtyard meetings, and various campaign programs.Since the election schedule was announced, campaigning has intensified further. Not only town areas, but even remote rural areas of the district are now covered with candidates’ banners, festoons

In [10]:
# -----------------------------
# Daily Star DataFrame
# -----------------------------
df_daily_star = pd.DataFrame({
    "source": ["The Daily Star"] * len(daily_star_texts),
    "language": ["en"] * len(daily_star_texts),
    "url": daily_star_urls,
    "article_text": daily_star_texts
})

# -----------------------------
# Prothom Alo DataFrame
# -----------------------------
df_prothom_alo = pd.DataFrame({
    "source": ["Prothom Alo"] * len(prothom_alo_texts),
    "language": prothom_alo_languages,   # "bn" or "en"
    "url": prothom_alo_urls,
    "article_text": prothom_alo_texts
})

# -----------------------------
# Combine both sources
# -----------------------------
df_all = pd.concat(
    [df_daily_star, df_prothom_alo],
    ignore_index=True
)

# -----------------------------
# Save FINAL dataset
# -----------------------------
df_all.to_csv("../Data/articles.csv", index=False)

print("✅ Web scraping complete.")
print("Total articles saved:", len(df_all))
print("Columns:", list(df_all.columns))
print("\nSource counts:")
print(df_all["source"].value_counts())
print("\nLanguage counts:")
print(df_all["language"].value_counts())


✅ Web scraping complete.
Total articles saved: 30
Columns: ['source', 'language', 'url', 'article_text']

Source counts:
source
Prothom Alo       20
The Daily Star    10
Name: count, dtype: int64

Language counts:
language
en    18
bn    12
Name: count, dtype: int64
